In [4]:
import os
import sys
from dataclasses import dataclass

import optuna
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../.."))

import src.utils.run_optuna as op
from src.utils.get_objective import get_objective

In [5]:
# === Configuration === (you edit here) ===
load_dotenv(dotenv_path="../../.env")


@dataclass
class Config:
    # Data / CV
    model_name: str = "xgb"
    data_id: str = "002"
    n_folds: int = 5
    seed: int = 42
    fold_idx: int = 0

    # Optuna
    n_trials: int = 10
    direction: str = "maximize"
    sampler: str = "tpe"  # tpe / random
    pruner: str = "median"  # median / none

    # Storage
    storage: str = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"


cfg = Config()

opts = {
    "earlys_stopping_rounds": 5,
    "max_epochs": 20,
    "min_epochs": 4,
}

# W&B
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc


True

In [6]:
# === Build & Run (frozen) ===
# sampler / pruner factory
def build_sampler(name, seed):
    if name == "tpe":
        return optuna.samplers.TPESampler(n_startup_trials=15, seed=seed)
    elif name == "random":
        return optuna.samplers.RandomSampler(seed=seed)
    else:
        raise ValueError(f"unknown sampler: {name}")


def build_pruner(name):
    if name == "median":
        return optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1000)
    elif name == "none":
        return optuna.pruners.NopPruner()
    else:
        raise ValueError(f"unknown pruner: {name}")


create_objective = get_objective(cfg.model_name)
objective = create_objective(
    cfg.data_id,
    seed=cfg.seed,
    n_folds=cfg.n_folds,
    fold_idx=cfg.fold_idx,
    wandb_project=wandb_project,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    opts=opts
)

sampler = build_sampler(cfg.sampler, cfg.seed)
pruner = build_pruner(cfg.pruner)

op.run_optuna_search(
    objective,
    n_trials=cfg.n_trials,
    n_jobs=1,
    direction=cfg.direction,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    storage=cfg.storage,
    sampler=sampler,
    pruner=pruner
)

[I 2025-10-01 22:05:20,833] A new study created in RDB with name: xgb-002


  0%|          | 0/10 [00:00<?, ?it/s]

['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Fold Col: 5fold-s42
Free CPU Mem: 14.92 GB
Free GPU Mem: 6.69 GB
[0]	train-auc:0.94750	valid-auc:0.94715
[100]	train-auc:0.96141	valid-auc:0.96108
[200]	train-auc:0.96372	valid-auc:0.96312
[300]	train-auc:0.96575	valid-auc:0.96491
[400]	train-auc:0.96749	valid-auc:0.96631
[500]	train-auc:0.96866	valid-auc:0.96717
[600]	train-auc:0.96962	valid-auc:0.96786
[700]	train-auc:0.97041	valid-auc:0.96840
[800]	train-auc:0.97096	valid-auc:0.96871
[900]	train-auc:0.97151	valid-auc:0.96902
[1000]	train-auc:0.97206	valid-auc:0.96933
[1100]	train-auc:0.97249	valid-auc:0.96952
[1200]	train-auc:0.97290	valid-auc:0.96971
[1300]	train-auc:0.97330	valid-auc:0.96987
[1400]	train-auc:0.97365	valid-auc:0.97001
[1500]	train-auc:0.97400	valid-auc:0.97014
[1600]	train-auc:0.97433	valid-auc:0.97025
[1700]	train-auc:0.97461	valid-auc:0.97035
[1800]	train-auc:0.97488	valid-auc:0.97044
[1900]	train-auc:0.97516	valid-auc:0

iter_f1,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇████
train/f1/auc,▁▂▃▃▃▃▄▄▄▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇████████
valid/f1/auc,▁▃▆▆▆▆▇▇▇▇▇▇████████████████████████████
auc_f1,0.97132
best_iter_f1,6479
iter_f1,6979
runtime_f1,1.57742
train/f1/auc,0.98413
valid/f1/auc,0.97129


[I 2025-10-01 22:06:59,137] Trial 0 finished with value: 0.9713247753907998 and parameters: {'learning_rate': 0.02, 'max_depth': 8, 'min_child_weight': 95.07143064099162, 'colsample_bytree': 0.592797576724562, 'subsample': 0.7394633936788146, 'reg_alpha': 0.0007482139197236472, 'reg_lambda': 0.000602521573620386}. Best is trial 0 with value: 0.9713247753907998.


...

iter_f1,▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇████
train/f1/auc,▁▁▂▂▂▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇████████
valid/f1/auc,▁▅▆▆▇▇▇▇▇▇▇▇████████████████████████████
auc_f1,0.97184
best_iter_f1,2734
iter_f1,3234
runtime_f1,1.35659
train/f1/auc,0.98618
valid/f1/auc,0.97181


[I 2025-10-01 22:18:52,581] Trial 8 finished with value: 0.9718366076612472 and parameters: {'learning_rate': 0.02, 'max_depth': 10, 'min_child_weight': 18.485445552552704, 'colsample_bytree': 0.6878338511058234, 'subsample': 0.8100531293444458, 'reg_alpha': 18.328605869327664, 'reg_lambda': 2.9794544625913595}. Best is trial 7 with value: 0.9712340110046801.


['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Fold Col: 5fold-s42
Free CPU Mem: 14.98 GB
Free GPU Mem: 6.59 GB
[0]	train-auc:0.94530	valid-auc:0.94522
[100]	train-auc:0.96178	valid-auc:0.96122
[200]	train-auc:0.96469	valid-auc:0.96380
[300]	train-auc:0.96691	valid-auc:0.96573
[400]	train-auc:0.96841	valid-auc:0.96685
[500]	train-auc:0.96951	valid-auc:0.96758
[600]	train-auc:0.97038	valid-auc:0.96813
[700]	train-auc:0.97107	valid-auc:0.96854
[800]	train-auc:0.97170	valid-auc:0.96886
[900]	train-auc:0.97230	valid-auc:0.96916
[1000]	train-auc:0.97279	valid-auc:0.96939
[1100]	train-auc:0.97324	valid-auc:0.96957
[1200]	train-auc:0.97367	valid-auc:0.96974
[1300]	train-auc:0.97409	valid-auc:0.96990
[1400]	train-auc:0.97448	valid-auc:0.97003
[1500]	train-auc:0.97483	valid-auc:0.97013
[1600]	train-auc:0.97520	valid-auc:0.97023
[1700]	train-auc:0.97557	valid-auc:0.97034
[1800]	train-auc:0.97591	valid-auc:0.97044
[1900]	train-auc:0.97620	valid-auc:0

iter_f1,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█
train/f1/auc,▁▂▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████
valid/f1/auc,▁▅▇▇▇▇▇▇▇███████████████████████████████
auc_f1,0.97097
best_iter_f1,5068
iter_f1,5568
runtime_f1,1.61046
train/f1/auc,0.98407
valid/f1/auc,0.97096


✅ Message sent.
✅ Document sent.
✅ Document sent.
✅ Document sent.
[I 2025-10-01 22:20:38,946] Trial 9 finished with value: 0.9709726034646942 and parameters: {'learning_rate': 0.02, 'max_depth': 10, 'min_child_weight': 92.18742350231169, 'colsample_bytree': 0.3353970008207678, 'subsample': 0.578393144967658, 'reg_alpha': 0.00017921154573376048, 'reg_lambda': 0.004233032996527599}. Best is trial 9 with value: 0.9709726034646942.
✅ Message sent.
